# CCC・投下資本回転率・税引後ROIC：実装と追加効果の初期検証

2026-11-03。EDINET DBを利用し、J-Quantsは現行プランのままです。本番の特徴量・パラメータ・モデルは変更していません。

## 比較結果の要点

以下の差は、値が大きいほどよい主指標（分類：Average Precision、ランキング：日付/組内NDCG@5）の差です。
指標の単位が異なるので、異なるモデルの差の大きさを直接比較しません。


In [1]:
from pathlib import Path
import json,pandas as pd,numpy as np
from IPython.display import display,HTML
P=Path.cwd()
def table(df):display(HTML(df.to_html(index=False,max_rows=None,float_format=lambda x:f'{x:,.5f}')))
res=pd.read_csv(P/'results.csv');plan=json.loads((P/'experiment_plan.json').read_text());inputs=json.loads((P/'inputs.json').read_text())
names={'class10':'分類 +10%','class15':'分類 +15%','ltr':'日次ランキング','ltr5':'直近5営業日ランキング'}
allres=res[res.subset.eq('all')]
wide=allres.pivot(index='task',columns='arm',values='primary').reindex(names)
summary=pd.DataFrame({'モデル':[names[x] for x in wide.index],'A 既存':wide.A,'Q 品質・欠損のみ':wide.Q,'B 財務値追加':wide.B,'P 値入替対照':wide.P,'B−A':wide.B-wide.A,'B−Q':wide.B-wide.Q,'B−P':wide.B-wide.P}).reset_index(drop=True)
table(summary)

モデル,A 既存,Q 品質・欠損のみ,B 財務値追加,P 値入替対照,B−A,B−Q,B−P
分類 +10%,0.23186,0.23121,0.23260,0.22925,0.00074,0.00140,0.00335
分類 +15%,0.18912,0.19113,0.18711,0.18921,-0.00201,-0.00402,-0.00210
日次ランキング,0.59575,0.58878,0.59143,0.59708,-0.00432,0.00265,-0.00565
直近5営業日ランキング,0.51908,0.50868,0.51554,0.52422,-0.00355,0.00686,-0.00868


今回の全評価行では、BがAを上回ったのは+10%分類だけでした。その差は約0.00074、品質・欠損だけのQとの差は約0.00140です。
種を変えると全モデルでB−Aの符号が変わり、ROICの付いた行の分類でもBはQを上回りませんでした。
**財務の数値自体の安定した追加効果は確認できず、本番への追加は保留**とする結果です。

B−Qが財務の数値を加えた差、B−Pが数値の対応関係を壊した対照との差です。B−Aだけが正でも、財務の数値が有効だったとは断定しません。

これは**LightGBMによる、1つの時系列ホールドアウトの探索的比較**です。3つの乱数種の予測を平均しましたが、独立した3つの評価期間ではありません。
Aは既存特徴量・既存パラメータを同じ最新マートで再学習した参照であり、保存済み本番モデルの予測をそのまま評価したものではありません。
このホールドアウトは過去の研究でも検討に使われています。またEDINET DBの過去訂正の完全な履歴は未確認です。
したがって、改善が見えても本番採用の証明にはしません。XGBoost・CatBoostの追加比較や再チューニングも今回の対象外です。

## 実装したもの

- `research/capital_efficiency.py`：値13列・品質12列。既存の本番360列とは独立した、明示的に呼び出す処理。
- 日本基準の借入金・社債・CP・リース等を集約。IFRSの合計と内訳の二重計上を避け、不完全な区分は比率を欠損にする。
- 非支配持分込み総資本か、親会社持分だけかが曖昧なIFRS行を識別。全社一律の非支配持分加算はしない。
- 補正には根拠書類と提出日が必須。翌年有報の前期末比較値も、その有報の提出翌日からしか使わない。
- 元のrawを上書きせず、後日の補正を利用開始イベントとして保存。最新年度の値が欠損でも、完全な古い年度に戻さない。
- 銀行・保険・証券商品先物・その他金融、業種不明を除外。期首末平均は同一基準・連続年度・通常期間・既知の前期に限定。

原本照合済みの3社6時点の分母を使った補正例は、別途照合した結果と一致しました。負債・純資産の照合状態は、PL/CF全項目の監査済み状態を意味しません。

## 算出候補率

分母は学習マート全22,110行。未取得銘柄や欠損行も落とさず、モデルに渡す段階では欠損のまま残します。
**値がある行には原本未照合の行も含みます。この表は原本確認済み率ではありません。**


In [2]:
cov=pd.read_csv(P/'coverage.csv')
table(cov.iloc[:13].rename(columns={'feature':'特徴量','rows':'値のある行','pct':'全学習行に対する割合(%)'}))
stats=pd.read_csv(P/'feature_summary.csv')
table(stats[['feature','count','unique_values','missing_pct','min','5%','50%','95%','max']])

特徴量,値のある行,全学習行に対する割合(%)
ce_dso,12946,58.55269
ce_dio,11327,51.23021
ce_dpo_cogs_proxy,12035,54.43238
ce_ccc_cogs_proxy,10936,49.46178
ce_nwc_to_sales,11555,52.26142
ce_gross_margin,13344,60.35278
ce_sga_ratio,14038,63.49163
ce_op_margin,13693,61.93125
ce_tax_rate,13111,59.29896
ce_nopat_margin_etr_proxy,12666,57.28630


feature,count,unique_values,missing_pct,min,5%,50%,95%,max
ce_dso,"12,946.00000",6957,41.44731,0.00482,6.99307,64.18707,136.32622,542.51965
ce_dio,"11,327.00000",6086,48.76979,0.00902,0.98093,55.36692,218.51148,"1,778.31788"
ce_dpo_cogs_proxy,"12,035.00000",6455,45.56762,0.00000,8.27833,44.66530,112.18777,"13,359.00000"
ce_ccc_cogs_proxy,"10,936.00000",5873,50.53822,"-4,024.57726",-10.58817,73.64457,256.34109,"1,275.05415"
ce_nwc_to_sales,"11,555.00000",6205,47.73858,-0.52494,0.00133,0.19014,0.52375,2.15207
ce_gross_margin,"13,344.00000",7276,39.64722,-1.01629,0.09855,0.26809,0.68926,1.00000
ce_sga_ratio,"14,038.00000",7681,36.50837,0.00206,0.05735,0.18152,0.55955,71.90549
ce_op_margin,"13,693.00000",7490,38.06875,-70.90549,0.00518,0.06750,0.23402,0.65346
ce_tax_rate,"13,111.00000",7169,40.70104,0.00000,0.16350,0.30503,0.45652,0.99485
ce_nopat_margin_etr_proxy,"12,666.00000",6927,42.71370,-0.41153,0.01035,0.04991,0.17080,0.52895


DSO＝平均売上債権/売上高×年度日数、DIO＝平均棚卸資産/売上原価×年度日数、DPO原価代用＝平均仕入債務/売上原価×年度日数。
CCC原価代用＝DSO＋DIO−DPO原価代用です。仕入高そのものは使っていません。

投下資本候補＝非支配持分込み純資産＋リース込み負債−現金等。
回転率＝売上高/平均投下資本、税引後ROIC近似＝営業利益×(1−会社全体の税率)/平均投下資本。
税率30%仮定版は実績税率版と別の指標です。現金等は全額控除の近似で、必要運転現金と余剰現金は分離していません。

原本で確認した任天堂のリース、NTTの長期借入債務と総資本は提出後の行だけ補正しています。
補正前の歴史的な行を一括で書き換えていません。

## 訓練と評価の分離

訓練終了は従来の直近12ヶ月ホールドアウトから決まる日付を基点としました。
各会社の実際の日足で20営業日目の結果確定日を求め、既存パラメータの探索に使われた訓練行も評価開始より前に確定するよう、必要なら評価開始を後へ動かしています。
訓練行・評価行はすべての腕で同じです。


In [3]:
split=json.loads((P/'split_checks.json').read_text());table(pd.DataFrame(split))
table(pd.DataFrame([{'項目':k,'値':plan[k]} for k in ['train_end','latest_parameter_training_outcome','test_start','test_end','seeds','trees','re_tuned','rolling_feature_date']]))

task,train_rows,test_rows,train_max_date,train_max_outcome,test_min_date,test_max_date
class10,18150,3654,2025-09-03 00:00:00,2025-10-03 00:00:00,2025-10-06 00:00:00,2026-10-02 00:00:00
class15,18150,3654,2025-09-03 00:00:00,2025-10-03 00:00:00,2025-10-06 00:00:00,2026-10-02 00:00:00
ltr,18150,3654,2025-09-03 00:00:00,2025-10-03 00:00:00,2025-10-06 00:00:00,2026-10-02 00:00:00
ltr5,90476,18247,2025-09-03 00:00:00,2025-10-03 00:00:00,2025-10-06 00:00:00,2026-10-02 00:00:00


項目,値
train_end,2025-09-03 00:00:00
latest_parameter_training_outcome,2025-10-03 00:00:00
test_start,2025-10-04 00:00:00
test_end,2026-10-02 00:00:00
seeds,"[42, 43, 44]"
trees,200
re_tuned,False
rolling_feature_date,candidate date; never refreshed at the pool decision date


LTR5は、判断日までの直近5営業日の候補を同じ組にします。既存特徴量と今回の財務指標は**候補日の値で凍結**し、判断日の新しい有報へ差し替えません。
値動き4列と判断日からの収益は、本番のLTR5処理を再利用し、日足からの直接再計算・同日候補との一致を検査しました。

## 4つの比較条件

| 腕 | 特徴量 |
|---|---|
| A | 既存360列。LTR5は値動き4列を加えた364列 |
| Q | A＋品質12列＋財務値13列の個別の欠損フラグ |
| B | Q＋財務値13列 |
| P | Q＋同じ日付内で観測値だけを入れ替えた財務値13列。各列の欠損位置は維持 |

各腕で同じ既存パラメータ・木200本・種42/43/44を使用。再探索による差を混ぜない比較です。
分類では本番と同じく、訓練ラベルの負例数/正例数で不均衡を補正します。各腕の訓練行が同じなので補正値も同じです。
値の追加は、CCC単独またはROIC単独の効果ではなく、分解要素を含む13列の集合の効果です。

## 全評価行での成績

収益の集計は翌営業日寄りから20営業日目終値までの単純収益で、取引コスト・同時保有制約等を含みません。
分類の上位10%は評価期間全体のスコア上位10%、ランキングの上位1/5件は候補5件以上の日の集計です。


In [4]:
table(allres.rename(columns={'task':'モデル','arm':'腕'}))

モデル,腕,subset,features,rows,days,positive_rate,primary,roc_auc,top_decile_precision,top_decile_close_return_pct,eligible_days,top1_close_return_pct,top5_close_return_pct
class10,A,all,360,3654,241,0.21018,0.23186,0.55635,0.20492,0.27414,NaN,NaN,NaN
class10,Q,all,385,3654,241,0.21018,0.23121,0.55796,0.22678,1.04285,NaN,NaN,NaN
class10,B,all,398,3654,241,0.21018,0.23260,0.55918,0.21311,1.20153,NaN,NaN,NaN
class10,P,all,398,3654,241,0.21018,0.22925,0.55083,0.21038,0.08377,NaN,NaN,NaN
class15,A,all,360,3654,241,0.13574,0.18912,0.63517,0.21311,2.18677,NaN,NaN,NaN
class15,Q,all,385,3654,241,0.13574,0.19113,0.63356,0.20219,2.36071,NaN,NaN,NaN
class15,B,all,398,3654,241,0.13574,0.18711,0.63102,0.20492,2.05360,NaN,NaN,NaN
class15,P,all,398,3654,241,0.13574,0.18921,0.63534,0.19399,1.59692,NaN,NaN,NaN
ltr,A,all,360,3654,241,NaN,0.59575,NaN,NaN,NaN,212.00000,2.78275,1.99719
ltr,Q,all,385,3654,241,NaN,0.58878,NaN,NaN,NaN,212.00000,1.15785,1.77719


## ROICの値がある評価行だけでの成績

各腕で同じ部分集合を使います。未取得銘柄を含む全行の成績を置き換えるものではありません。


In [5]:
table(res[res.subset.eq('roic_available')])

task,arm,subset,features,rows,days,positive_rate,primary,roc_auc,top_decile_precision,top_decile_close_return_pct,eligible_days,top1_close_return_pct,top5_close_return_pct
class10,A,roic_available,360,1767,231,0.20487,0.24615,0.58296,0.20904,0.43318,NaN,NaN,NaN
class10,Q,roic_available,385,1767,231,0.20487,0.24815,0.58775,0.24294,0.68397,NaN,NaN,NaN
class10,B,roic_available,398,1767,231,0.20487,0.24676,0.58555,0.22034,1.25412,NaN,NaN,NaN
class10,P,roic_available,398,1767,231,0.20487,0.24581,0.58000,0.23729,1.18761,NaN,NaN,NaN
class15,A,roic_available,360,1767,231,0.13186,0.20630,0.64858,0.23164,2.29288,NaN,NaN,NaN
class15,Q,roic_available,385,1767,231,0.13186,0.20723,0.64852,0.23729,3.63495,NaN,NaN,NaN
class15,B,roic_available,398,1767,231,0.13186,0.20484,0.64811,0.23164,2.56004,NaN,NaN,NaN
class15,P,roic_available,398,1767,231,0.13186,0.20215,0.64839,0.20904,1.87576,NaN,NaN,NaN
ltr,A,roic_available,360,1767,231,NaN,0.62860,NaN,NaN,NaN,139.00000,1.25492,0.87175
ltr,Q,roic_available,385,1767,231,NaN,0.63043,NaN,NaN,NaN,139.00000,0.93811,1.13510


## 四半期ごとの差

四半期内で100行以上・20日以上ある期間だけ集計。最後の数日だけの四半期は比較窓に数えません。


In [6]:
w=pd.read_csv(P/'quarter_results.csv')
z=w.pivot(index=['task','quarter'],columns='arm',values='primary')
table(z.assign(B_minus_A=z.B-z.A,B_minus_Q=z.B-z.Q,B_minus_P=z.B-z.P).reset_index())

arm,task,quarter,A,B,P,Q,B_minus_A,B_minus_Q,B_minus_P
,class10,2025Q4,0.26511,0.26608,0.26011,0.26730,0.00096,-0.00122,0.00596
,class10,2026Q1,0.27272,0.27676,0.26670,0.27480,0.00404,0.00196,0.01005
,class10,2026Q2,0.24563,0.24770,0.24258,0.24503,0.00207,0.00267,0.00512
,class10,2026Q3,0.19022,0.18493,0.19415,0.18890,-0.00529,-0.00397,-0.00921
,class15,2025Q4,0.18458,0.17944,0.18583,0.18411,-0.00513,-0.00467,-0.00638
,class15,2026Q1,0.24734,0.24404,0.24429,0.25091,-0.00330,-0.00687,-0.00025
,class15,2026Q2,0.21954,0.21682,0.21639,0.22182,-0.00271,-0.00500,0.00043
,class15,2026Q3,0.13658,0.13311,0.13913,0.13451,-0.00347,-0.00141,-0.00602
,ltr,2025Q4,0.59430,0.59075,0.59742,0.59223,-0.00355,-0.00148,-0.00667
,ltr,2026Q1,0.54747,0.55479,0.56138,0.53939,0.00732,0.01540,-0.00658


## 乱数種による差

下表は種ごとの単体モデルの主指標。要点の表は3種の予測を平均した後の主指標なので、下表の単純平均とは一致しない場合があります。


In [7]:
s=pd.read_csv(P/'seed_results.csv')
z=s.pivot(index=['task','seed'],columns='arm',values='primary')
table(z.assign(B_minus_A=z.B-z.A,B_minus_Q=z.B-z.Q,B_minus_P=z.B-z.P).reset_index())

arm,task,seed,A,B,P,Q,B_minus_A,B_minus_Q,B_minus_P
,class10,42,0.23168,0.22993,0.22686,0.22805,-0.00175,0.00188,0.00307
,class10,43,0.23191,0.23459,0.22883,0.22816,0.00268,0.00643,0.00576
,class10,44,0.23121,0.23292,0.23001,0.23863,0.00171,-0.00572,0.00290
,class15,42,0.18683,0.18475,0.18509,0.19110,-0.00208,-0.00635,-0.00034
,class15,43,0.18821,0.18941,0.19031,0.19042,0.00120,-0.00101,-0.00090
,class15,44,0.19120,0.18584,0.19034,0.18990,-0.00536,-0.00406,-0.00450
,ltr,42,0.59420,0.59599,0.59266,0.58931,0.00178,0.00668,0.00333
,ltr,43,0.59581,0.59506,0.59332,0.59160,-0.00075,0.00346,0.00174
,ltr,44,0.60384,0.59490,0.60204,0.59157,-0.00893,0.00333,-0.00713
,ltr5,42,0.51625,0.51336,0.52019,0.50622,-0.00289,0.00714,-0.00683


## 再現性・検証の記録

入力データ、年次財務、会社対応表、FY期間の各年ファイル、原本補正ファイル、実装・実験コード、パラメータファイルのSHA256を `inputs.json` に保存。表示用Notebookは同じフォルダの集計CSV/JSONだけで再実行できます。
`experiment_plan.json` はモデルの評価前に保存しました。元データ・銘柄別の財務値・予測行は公開リポジトリに含めていません。


In [8]:
complete=json.loads((P/'completed.json').read_text())
checks=json.loads((P/'verification.json').read_text())
table(pd.DataFrame([{'項目':k,'値':v} for k,v in {**complete,**checks}.items() if not isinstance(v,(dict,list))]))

項目,値
tasks,4
arms,4
seeds,3
fits,48
production_changed,False
premium_changed,False
broad_test_count,2098
broad_tests_skipped,1
broad_test_result,pass
final_affected_test_count,27


## 残る制約

負債明細の完全性、リースの会計基準差、過去訂正の履歴、全銘柄の原本照合は未完了です。現在の比率は品質条件を付けた研究用候補です。
J-Quants Premiumへの変更、EDINET取得頻度の変更、本番特徴量への採用や本番再学習はしていません。
初期比較の結果と、財務指標自体が重要な投資指標であることは分けて扱います。
